# Input → output, one document

What a user actually gets: point the pipeline at one sample number, and
compare what went in against what came out. No scoring, no charts — this
notebook only answers "what does the input look like, what does the output
look like".


## Pick a sample

`dmpbridge` is installed as a real Python package (`pip install -e .`) —
importable from any directory, not just this project folder. Everything below
is imported from it, the same as any other installed library.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json

import dmpbridge
print(f'dmpbridge {dmpbridge.__version__}, installed at {Path(dmpbridge.__file__).parent}')

from dmpbridge.core import paths as P

SAMPLE    = 1                  # <- change this to look at a different document
MODEL     = 'llama3.1:8b'
EXTRACTOR = 'pdfplumber'

tag = P.make_tag(MODEL, EXTRACTOR)
pdf_path = Path('data/input/pdfs') / f'sample{SAMPLE}.pdf'
print(f'PDF      : {pdf_path}')
print(f'model    : {MODEL}')
print(f'extractor: {EXTRACTOR}')


dmpbridge 0.1.0, installed at C:\Users\Nahid\dmpbridge\dmpbridge
PDF      : data\input\pdfs\sample1.pdf
model    : llama3.1:8b
extractor: pdfplumber


## Input

The pipeline never reads the PDF's meaning directly — the first step turns it
into a flat list of text blocks. This is the actual input the model sees, before
any labeling happens.


In [2]:
extracted_path = P.extracted_path(EXTRACTOR, SAMPLE)
blocks = json.loads(extracted_path.read_text(encoding='utf-8'))

print(f'{len(blocks)} text blocks extracted from {pdf_path.name}\n')
for b in blocks[:6]:
    tag_mark = '[heading]' if b.get('is_bold') else '         '
    print(f"  {tag_mark} {b['text'][:80]!r}")
if len(blocks) > 6:
    print(f'  ... and {len(blocks) - 6} more blocks')


78 text blocks extracted from sample1.pdf

  [heading] 'DATA MANAGEMENT AND SHARING PLAN'
  [heading] 'Element 1: Data Type:'
  [heading] 'A. Types and amount of scientific data expected to be generated in the project:'
            'This secondary data analysis project will analyze deidentified data from 48,218 '
            'and the publicly available NHANES cohorts (wrist NHANES 2011-2014; hip/counts-NH'
            'The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the iWATC'
  ... and 72 more blocks


## Output

After labeling, structuring, and filling in blank questions from their section
heading, this is the final document — the thing a user actually opens.


In [3]:
final_path = P.final_path(tag, SAMPLE)
doc = json.loads(final_path.read_text(encoding='utf-8'))
template = doc['narrative']['template']

print(f'TITLE: {template["title"]}\n')
for i, section in enumerate(template['section'], 1):
    print(f'{i}. {section["title"]}')
    if section.get('description'):
        print(f'   description: {section["description"][:80]}')
    for q in section['question']:
        answer = q['answer']['json']['answer']
        print(f'   Q: {q["text"][:80]}')
        print(f'   A: {answer[:100]}{"..." if len(answer) > 100 else ""}')
    print()


TITLE: DATA MANAGEMENT AND SHARING PLAN

1. Element 1: Data Type:
   Q: A. Types and amount of scientific data expected to be generated in the project:
   A: This secondary data analysis project will analyze deidentified data from 48,218 participants from ei...

2. B. Scientific data that will be preserved and shared, and the rationale for doing so:
   Q: B. Scientific data that will be preserved and shared, and the rationale for doin
   A: As this is a secondary data analysis project, we will only be able to publicly share in the UC San D...

3. C. Metadata, other relevant data, and associated documentation:
   Q: C. Metadata, other relevant data, and associated documentation:
   A: In addition to the data described above, code and models will be included in the repository and in o...

4. Element 2: Related Tools, Software and/or Code:
   Q: Element 2: Related Tools, Software and/or Code:
   A: Data will be analyzed with custom code by our statistical and computer science team. ActiGr

That's the whole transformation: a PDF becomes a flat list of text blocks
(input), and after labeling, structuring, and rule-filling, becomes a nested
title/section/question/answer document (output). Change `SAMPLE`, `MODEL`, or
`EXTRACTOR` above and re-run to see a different document.
